<a href="https://colab.research.google.com/github/alyssaplayer/BU_OMDS_APlayerRepo/blob/main/DX799_Capstone_OList_(Marketing)_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Load Required Libraries


In [9]:
pip install pandas matplotlib seaborn numpy kagglehub


Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 10.8 MB 7.8 MB/s eta 0:00:01
     |████████████████████████████████| 7.8 MB 83.4 MB/s eta 0:00:01
     |████████████████████████████████| 294 kB 23.7 MB/s eta 0:00:01
     |████████████████████████████████| 5.3 MB 29.9 MB/s eta 0:00:01
     |████████████████████████████████| 349 kB 48.2 MB/s eta 0:00:01
     |████████████████████████████████| 510 kB 54.1 MB/s eta 0:00:01
     |████████████████████████████████| 2.9 MB 73.0 MB/s eta 0:00:01
     |████████████████████████████████| 249 kB 17.1 MB/s eta 0:00:01
     |████████████████████████████████| 122 kB 19.9 MB/s eta 0:00:01
     |████████████████████████████████| 4.7 MB 26.6 MB/s eta 0:00:01
     |████████████████████████████████| 64 kB 15.7 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the 

In [7]:
pip install kagglehub


Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 68 kB 7.1 MB/s eta 0:00:011
     |████████████████████████████████| 78 kB 15.6 MB/s eta 0:00:01
     |████████████████████████████████| 64 kB 18.3 MB/s eta 0:00:01
     |████████████████████████████████| 174 kB 27.5 MB/s eta 0:00:01
     |████████████████████████████████| 65 kB 22.2 MB/s eta 0:00:01
     |████████████████████████████████| 131 kB 21.6 MB/s eta 0:00:01
     |████████████████████████████████| 134 kB 60.1 MB/s eta 0:00:01
     |████████████████████████████████| 299 kB 33.4 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [4]:
import kagglehub
import pandas as pd
import os
import glob
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

ModuleNotFoundError: No module named 'kagglehub'

In [ ]:
#Loading the Dataset
#Brazilian E-Commerce Public Dataset by Olist
#Kaggle

# Download latest version
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print("Path to dataset files:", path)

df_raw = path

#to save the dataset:
#df.to_csv("olist_df.csv", index=False)

dataset_path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
print("Dataset downloaded to:", dataset_path)

# List all downloaded CSV files
csv_files = [f for f in os.listdir(dataset_path) if f.endswith('.csv')]
print(f"\nFiles available ({len(csv_files)} CSVs):")
for f in sorted(csv_files):
    print(f"  {f}")

# ── Cell 3: Load each CSV into a named DataFrame ──────────────────────────────
import pandas as pd

orders        = pd.read_csv(os.path.join(dataset_path, "olist_orders_dataset.csv"))
order_items   = pd.read_csv(os.path.join(dataset_path, "olist_order_items_dataset.csv"))
order_reviews = pd.read_csv(os.path.join(dataset_path, "olist_order_reviews_dataset.csv"))
customers     = pd.read_csv(os.path.join(dataset_path, "olist_customers_dataset.csv"))
products      = pd.read_csv(os.path.join(dataset_path, "olist_products_dataset.csv"))
sellers       = pd.read_csv(os.path.join(dataset_path, "olist_sellers_dataset.csv"))
payments      = pd.read_csv(os.path.join(dataset_path, "olist_order_payments_dataset.csv"))
geo           = pd.read_csv(os.path.join(dataset_path, "olist_geolocation_dataset.csv"))
product_names = pd.read_csv(os.path.join(dataset_path, "product_category_name_translation.csv"))

print("All datasets loaded successfully.")

In [ ]:
#merging all the pertinent datasets in the relational database together
import pandas as pd

# 2. Build the master DataFrame
# Start with orders and add items
df = pd.merge(orders, order_items, on='order_id', how='left')

# Add the review data (adding in the customer satisfaction data)
df = pd.merge(df, order_reviews, on='order_id', how='left')

# Add the customer location/demographic data
df = pd.merge(df, customers, on='customer_id', how='left')

# Check the resulting dataset
print(df.info())

In [ ]:
#EDA
display(df.head(), df.describe())



In [ ]:
#EDA - Checking Available Data in the Dataset
# Unique values per column
print("Unique values per column:")
display(df.nunique().to_frame("unique_count"))

# Null values per column
print("\nNull values per column:")
display(df.isnull().sum().to_frame("null_count"))

# Duplicate rows
print(f"\nNumber of duplicate rows: {df.duplicated().sum()}")

# Feature Engineering


In [ ]:
#1. Date/Time Columns
date_cols = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date', 'shipping_limit_date',
    'review_creation_date', 'review_answer_timestamp'
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col])

# 2. Numeric Features from Dates
# Days from purchase to actual delivery (key satisfaction driver)
df['delivery_days'] = (
    df['order_delivered_customer_date'] - df['order_purchase_timestamp']
).dt.total_seconds() / 86400

#86400 seconds = 1 day

# Early or late dependent on expected delivery date (negative = early, positive = late)
df['delivery_delay_days'] = (
    df['order_delivered_customer_date'] - df['order_estimated_delivery_date']
).dt.total_seconds() / 86400

# Days from purchase to approval
df['approval_days'] = (
    df['order_approved_at'] - df['order_purchase_timestamp']
).dt.total_seconds() / 86400

# How quickly the seller handed to carrier
df['carrier_handoff_days'] = (
    df['order_delivered_carrier_date'] - df['order_approved_at']
).dt.total_seconds() / 86400

# How long the customer waited for a review response
df['review_response_days'] = (
    df['review_answer_timestamp'] - df['review_creation_date']
).dt.total_seconds() / 86400

# 3. Encode order_status (ordinal — reflects fulfilment progress)
status_order = {
    'created': 0, 'approved': 1, 'processing': 2, 'invoiced': 3,
    'shipped': 4, 'delivered': 5, 'unavailable': -1, 'canceled': -2
}
df['order_status_enc'] = df['order_status'].map(status_order)

#  4. Encode customer_state (frequency encoding — avoids 27 dummies)
state_freq = df['customer_state'].value_counts(normalize=True)
df['customer_state_freq'] = df['customer_state'].map(state_freq)

#  5. Has review comment? (binary signal)
df['has_comment'] = df['review_comment_message'].notna().astype(int)

#  6. Drop rows missing the target (review_score)
df_model = df.dropna(subset=['review_score']).copy()

#  7. Final feature matrix for modelling
feature_cols = [
    'price', 'freight_value', 'order_item_id',
    'delivery_days', 'delivery_delay_days',
    'approval_days', 'carrier_handoff_days', 'review_response_days',
    'order_status_enc', 'customer_state_freq', 'has_comment'
]



In [ ]:
#Correlation Matrix
import seaborn as sns
corr_matrix = df_model[feature_cols + ['review_score']].corr(numeric_only=True)

plt.figure(figsize=(12, 9))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', center=0,
            linewidths=0.5, annot_kws={"size": 8})
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()


In [ ]:
#Train / Test / Split
from sklearn.model_selection import train_test_split

df_model = df_model.dropna(subset=feature_cols).copy()

X = df_model[feature_cols]
y = df_model['review_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Modelling dataset: {X.shape[0]:,} rows × {X.shape[1]} features")
print(f"Train: {X_train.shape[0]:,} rows | Test: {X_test.shape[0]:,} rows")
print(f"\nTarget distribution:\n{y.value_counts().sort_index()}")


# W1: Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(y_test, y_pred, alpha=0.4, color='steelblue', edgecolors='white', linewidth=0.3)
axes[0].plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()], 'r--', linewidth=2, label='Perfect Fit')
axes[0].set_xlabel('Actual Review Score')
axes[0].set_ylabel('Predicted Review Score')
axes[0].set_title('Actual vs. Predicted Review Score')
axes[0].legend()
axes[0].annotate(f'R² = {r2_score(y_test, y_pred):.3f}',
                 xy=(0.05, 0.92), xycoords='axes fraction', fontsize=11, color='darkred')

# Feature coefficients
coef_df = pd.Series(model.coef_, index=feature_cols).sort_values()
axes[1].barh(coef_df.index, coef_df.values, color=['tomato' if v < 0 else 'steelblue' for v in coef_df.values])
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Linear Regression — Feature Coefficients')
axes[1].set_xlabel('Coefficient Value')

plt.tight_layout()
plt.show()

print(f"R²:   {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

# Week 2: Lasso, Ridge, and Elastic Net Regression

In [ ]:
#Lasso Regression
from sklearn.linear_model import Lasso

# Initialize the model (alpha acts as the lambda regularization parameter)
# By default, alpha=1.0
lasso_model = Lasso(alpha=1.0)

# Fit the model to your training data
lasso_model.fit(X_train, y_train)

# Make predictions
predictions = lasso_model.predict(X_test)

In [ ]:
#Ridge Regression
from sklearn.linear_model import Ridge

# Initialize the model
# By default, alpha=1.0
ridge_model = Ridge(alpha=1.0)

# Fit the model to your training data
ridge_model.fit(X_train, y_train)

# Make predictions
predictions = ridge_model.predict(X_test)

#use GridSearch CV to test lambda

In [ ]:
#Elastic Net Regression
from sklearn.linear_model import ElasticNet

# Initialize the model
# alpha controls the overall strength of the penalty
# l1_ratio controls the mix (l1_ratio=1.0 is pure Lasso, 0.0 is pure Ridge)
# Default l1_ratio is generally 0.5 (an even split)
elastic_net_model = ElasticNet(alpha=1.0, l1_ratio=0.5)

# Fit the model to your training data
elastic_net_model.fit(X_train, y_train)

# Make predictions
predictions = elastic_net_model.predict(X_test)

In [ ]:

for name, m in [("Lasso", lasso_model), ("Ridge", ridge_model), ("ElasticNet", elastic_net_model)]:
    preds = m.predict(X_test)
    print(f"{name:12s} | R²: {r2_score(y_test, preds):.4f} | RMSE: {np.sqrt(mean_squared_error(y_test, preds)):.4f}")

# W3: Forward / Backward Selection

In [ ]:
#forward and backward selection, PCR, and PLSR.

#W4

In [ ]:
#logistic regression and feature scaling

#W5: Support Vector Machines

In [ ]:
#or Week 5, include concepts such as support vector machines, the kernel trick, and regularization for support vector machine